# MIRI Drug Repositioning Screen — xpert-fda-v1

Single orchestrator for the numbered workflow stages (01-10). Run cells in order. Each stage is independently runnable from `scripts/` and produces its own artifact.

Environment: `conda activate arc` (see root `environment.yml`). Heavy stages (02 embeddings, 04 finetune, 05 predict) require a GPU.

## Stage 01 — Drug list

Generates the 3,127-drug FDA list from the ChEMBL source table. Output: `data/drug_list.csv`.

In [ ]:
!python scripts/01_drug_list.py

## Stage 02 — Drug embeddings

Arranges the frozen UniMol array + SMILES mapping and generates the HG biological embeddings. Idempotent: skips HG inference when an intact `data/embeddings/all_drugs_idx2HG.npy` exists. Output: `data/embeddings/`.

In [ ]:
!python scripts/02_drug_embeddings.py

# Force regeneration:
# !python scripts/02_drug_embeddings.py --force

## Stage 03 — Baseline transcriptome

Builds the workflow-owned consensus cardiomyocyte baseline from DTOXS control wells. Output: `data/baseline/cm_ctl_landmark.csv`.

In [ ]:
!python scripts/03_baseline_transcriptome.py

## Stage 04 — Finetune XPert on DTOXS

Runs the dtoxs finetune (workflow-owned weights). Output: `finetune/output/dtoxs_dtoxs/`.

In [ ]:
!python scripts/04_finetune_xpert.py

## Stage 05 — Perturbed transcriptomes

Predicts drug perturbation (deg) for all FDA drugs via XPert. Predict indexes the precomputed UniMol array by drug index, so finetune and predict consume identical features. Output: `data/deg/xpert_fda-cm_deg.csv`.

In [ ]:
!python scripts/05_perturbed_transcriptomes.py

## Stage 06 — Z-score

Z-score normalizes the predicted DEG per gene across drugs. Output: `data/deg-zs/xpert_fda-cm_deg-zs.csv`.

In [ ]:
!python scripts/06_zscore.py

## Stage 07 — Build MIRI reference DEG list

Consumes the per-GSE DEG lists in `disease_data/miri/{GSE}/deg/` and produces the integrated reference (`MIRI_up-down.json`, `MIRI_rank.json`).

In [ ]:
!python scripts/07_build_miri_deg_reference.py

## Stage 08 — Efficacy scores

Runs the three scorers (WCS+RGES, Zhang/ssCMap, pathway RGES) and computes the four z-components + integrated score. Output: `data/scores/1-scores.csv`.

In [ ]:
!python scripts/08_efficacy_scores.py

## Stage 09 — Reports (gpt-researcher)

Builds the deterministic report-gating artifacts and prints the prompts for the gpt-researcher steps (4-regulation, 5_efficacy_reports, 6-reposition_report). The evidence reports require API keys in `.env` and the gpt-researcher skill; run them via the agent workflow described in `prompts/` or pass `--run-gptr`.

In [ ]:
!python scripts/09_reports.py

## Stage 10 — Filter + rank

Applies the WCS<0 and critical-pathway filters and produces the final ranked drug list. Output: `data/ranked/ranked_drugs.csv`.

In [ ]:
!python scripts/10_filter_rank.py

## Pipeline summary

The final ranked drug list is `data/ranked/ranked_drugs.csv`. For parity checking against the pre-refactor pipeline, compare this output with the archived result in `miri_reposition/xpert-fda-v1_history/`.

In [ ]:
import pandas as pd
ranked = pd.read_csv('data/ranked/ranked_drugs.csv')
print(f'{len(ranked)} drugs passed the filters')
print(ranked[['drug_name', 'integrated']].head(10).to_string(index=False))